# Subscribing to the ToF topic

[Notebook 1](./1-tof-basics.ipynb) covered what a time-of-flight sensor measures, and [Notebook
  2](./2-duckiedrone-tof-architecture.ipynb) the chain that carries a reading to a ROS 2 topic.

This notebook writes the node at the end of that chain. The node subscribes to the downward
  sensor's topic and prints the newest reading once a second, so the value can be watched as the
  distance under the sensor changes.

Only the downward sensor runs by default, so this notebook uses that one. [Notebook
  4](./4-all-five-sensors.ipynb) starts the other four and points the same node at all five.

By the time a reading reaches the topic, the
  [driver](./2-duckiedrone-tof-architecture.ipynb#step-2) and the
  [bridge](./2-duckiedrone-tof-architecture.ipynb#step-3) have already converted it to meters,
  stamped it, and marked unusable readings. The node only has to ask ROS 2 for the messages and
  unpack them.

By the end of the notebook you should be able to answer:

- Why does `_make_callback` build a callback instead of the node passing one method?

- Why does the callback store the reading rather than print it?

- What does one printed line describe?

- Which printed values are not distances, and what should a consumer do with them?

## Getting to a terminal that can see the Duckiedrone

The topic exists on the Duckiedrone, so the commands in this notebook run in a terminal there.
  Four steps lead to one. A physical and a virtual Duckiedrone run the same containers, so every
  command from Step 2 onward is identical on the two; only the way the first shell is opened, in
  Step 1, differs.

### Step 0. Reach the Duckiedrone from the base station

<a id="step-0"></a>

On the base station, the computer with the [Duckietown
  Shell](https://docs.duckietown.com/ente/duckietown-manual/10-setup/02-software/duckietown-shell-dts-installation.html)
  installed, check that the Duckiedrone answers:

```bash
ping <robot>.local
```

where `<robot>` is the Duckiedrone's hostname. If it does not answer, the steps below fail too.
  See the [network troubleshooting
  tips](https://docs.duckietown.com/ente/duckietown-manual/06-troubleshooting/duckiebot-network-faqs.html)
  and the [Duckiedrone network
  configuration](https://docs.duckietown.com/ente/opmanual-dd24/40-duckiedrone-handling/70-network-configuration.html).

### Step 1. Open a shell on the Duckiedrone

<a id="step-1"></a>

On a physical Duckiedrone, connect with SSH:

```bash
ssh duckie@<robot>.local
```

The password is the one set when the SD card was flashed. If none was set, the default is
  `quackquack`.

SSH is not fully supported on a virtual Duckiedrone. From the base station, open its shell with:

```bash
dts duckiebot virtual connect <robot>
```

This opens a root shell on the virtual Duckiedrone, and `exit` returns to the base station.

### Step 2. Find the containers that publish the ToF

<a id="step-2"></a>

As [Notebook 2](./2-duckiedrone-tof-architecture.ipynb) described, each program on the
  Duckiedrone runs in its own container. List them:

```bash
docker ps --format '{{.Names}}'
```

Among them are the containers from Notebook 2:

```
driver-tof-bottom
ros2-tof-bottom
ros2-mavros
driver-camera
dashboard
...
```

`driver-tof-bottom` is the [driver](./2-duckiedrone-tof-architecture.ipynb#step-2) and
  `ros2-tof-bottom` the [bridge](./2-duckiedrone-tof-architecture.ipynb#step-3), while
  `ros2-mavros` forwards the range to the flight controller. Only the downward sensor's pair
  runs by default.

### Step 3. Open a shell inside the bridge

<a id="step-3"></a>

```bash
docker exec -it ros2-tof-bottom bash
```

[`docker exec`](https://docs.docker.com/reference/cli/docker/container/exec/) runs a command
  inside a running container, and `-it` makes it an interactive terminal. The prompt changes,
  and `exit` returns to the Duckiedrone's own shell. A warning about a missing colcon setup file
  is normal here and harmless.

The container already has ROS 2 set up, including the Duckiedrone's `ROS_DOMAIN_ID`, so `ros2`
  commands work straight away. A shell anywhere else needs ROS 2 installed, sourced, and set to
  the same domain first.

On a virtual Duckiedrone, the readings come from the
  [Duckiematrix](https://docs.duckietown.com/ente/duckietown-manual/50-duckiematrix/getting-started/duckiematrix-first-steps.html),
  so the topic carries data only while the Duckiematrix is running and the Duckiedrone is
  attached to an entity, as in the [general
  procedure](https://docs.duckietown.com/ente/opmanual-dd24/50-learning-experiences/lx-general-procedure.html).

## Look at the data first

The downward sensor publishes whether or not anything listens, so the topic can be inspected
  before any code exists. Three [`ros2
  topic`](https://docs.ros.org/en/jazzy/Tutorials/Beginner-CLI-Tools/Understanding-ROS2-Topics/Understanding-ROS2-Topics.html)
  commands do that.

**What exists:**

```bash
ros2 topic list | grep tof
```

```
/<robot>/bottom_tof_driver_node/range
```

That is the topic from [Notebook 2](./2-duckiedrone-tof-architecture.ipynb#step-3), with the
  Duckiedrone's hostname in front. A name in this list does not prove that anything publishes on
  it, because a subscription alone also adds the name. `ros2 topic info` counts the publishers:

```bash
ros2 topic info /<robot>/bottom_tof_driver_node/range
```

```
Type: sensor_msgs/msg/Range
Publisher count: 1
Subscription count: 3
```

The publisher is the bridge. The subscribers are MAVROS2, which forwards the readings to the
  flight controller, and two other Duckiedrone nodes that read the range, `altitude_node` and
  `state_estimator_node`. `ros2 topic info -v` lists each one by name.

**How often it arrives:**

```bash
ros2 topic hz /<robot>/bottom_tof_driver_node/range
```

```
average rate: 30.001
	min: 0.021s max: 0.044s std dev: 0.00265s window: 798
```

The downward sensor is configured for 30 Hz, and this output from a physical Duckiedrone
  matches. A virtual Duckiedrone can differ, so measure it rather than assume. Ctrl-C stops the
  command.

**What is inside one message:**

```bash
ros2 topic echo --once /<robot>/bottom_tof_driver_node/range
```

Compare the output with the message in [Notebook 1](./1-tof-basics.ipynb) and the field table in
  [Notebook 2](./2-duckiedrone-tof-architecture.ipynb#step-3). The node written next uses three
  of the fields: `range`, `header.stamp`, and `header.frame_id`.

## Activity 1: write the subscriber

<a id="activity-1"></a>

Open `packages/solution/solution/tof_subscriber.py`. Most of the file is already written: it
  declares the node's parameters, builds the full topic name for each sensor, creates the empty
  dictionary `self.latest_readings`, and starts a timer that calls `cb_log` once a second to
  print.

Two pieces are missing, each marked with a `TO IMPLEMENT` block that lists the names and
  expressions to use.

### In `__init__`, ask ROS 2 for the messages

A subscription is a standing request: it names a topic and a function, and from then on ROS 2
  calls that function with every message that arrives. The node never asks for a message itself.
  This is the node at the end of the chain in [Step 5 of Notebook
  2](./2-duckiedrone-tof-architecture.ipynb#step-5).

`create_subscription` takes four arguments, and two of them need explaining.

The callback argument is `self._make_callback(name)`, with parentheses: it runs once per sensor,
  when the subscription is created, and returns the function ROS 2 will call later. The next
  section explains why.

The last argument is a [Quality of Service
  (QoS)](https://docs.ros.org/en/jazzy/Concepts/Intermediate/About-Quality-of-Service-Settings.html)
  profile, which sets whether a lost message is sent again and how many messages wait in the
  queue. `qos_profile_sensor_data` does not resend and keeps a short queue. That fits a stream
  where a newer reading follows about 33 ms later, and it is the profile MAVROS2 uses on this
  topic. A plain queue depth of `10` also works, so this is a choice to match the rest of the
  stack, not a requirement. To compare the profiles of the publisher and the subscribers:

```bash
ros2 topic info -v /<robot>/bottom_tof_driver_node/range
```

### In `_make_callback`, unpack one message

`_make_callback` builds the callback rather than being one. ROS 2 passes a callback only the
  message, and in [Notebook 4](./4-all-five-sensors.ipynb) the same node subscribes to five
  sensors. The message's `header.frame_id`, `<robot>/tof/<sensor_name>`, names the sensor on a
  physical Duckiedrone, but a virtual Duckiedrone leaves it empty, so the node does not rely on
  it. Instead, each callback is built with its sensor's name: the inner function remembers
  `name` from the call that created it.

The inner function receives one `sensor_msgs/Range` and stores a `ToFReading` under
  `self.latest_readings[name]`, where `cb_log` finds it a second later. The timestamp arrives as
  two integers, seconds and nanoseconds, because one 64-bit float cannot hold the current time
  to the nanosecond. The `TO IMPLEMENT` block adds them into one float anyway, which rounds the
  time to about a quarter of a microsecond, far finer than printing needs.

The callback stores the reading and does not print it, for two reasons. Messages arrive at about
  30 Hz, far faster than anyone can read. And ROS 2 delivers each message by calling this
  function, so slow work inside it delays every message behind it. Printing from the timer keeps
  the two rates independent.

With both pieces written, the node is ready to run.

## Activity 2: run the node on the Duckiedrone

<a id="activity-2"></a>

The node has to run on the Duckiedrone, in a container next to the ones `docker ps` listed.
  `dts` does this without copying files by hand: it syncs the repository to the Duckiedrone,
  builds an image there, and runs it as a container called the **workbench**.

Both commands run on the base station, from the root folder of this learning experience. First,
  build the image:

```bash
dts code build -R <robot>
```

The first build takes a few minutes. Then start the workbench:

```bash
dts code workbench -R <robot>
```

The workbench sets up ROS 2 and this repository's packages, then runs the launch file, which
  reads the Duckiedrone's hostname from the container's environment. The node's output appears
  in the terminal:

```
[tof_listener_node-1] [INFO] [tof_listener_node]: Subscribing to /pdrone24/bottom_tof_driver_node/range  (sensor: bottom)
[tof_listener_node-1] [INFO] [tof_listener_node]: ranges=[bottom=0.018m]
```

The first line shows the topic the node subscribed to; its hostname and sensor name should match
  the topic inspected above. The `[tof_listener_node-1]` prefix is added by `ros2 launch`, which
  labels each process's output.

`Ctrl-C` stops the node. After any change to the code, run both commands again, as the [general
  procedure](https://docs.duckietown.com/ente/opmanual-dd24/50-learning-experiences/lx-general-procedure.html)
  describes.

### When nothing arrives

If the node keeps printing `Waiting for ToF data...`, it is running but receiving nothing. First
  check that both `TO IMPLEMENT` blocks from [Activity 1](#activity-1) are written: without them
  the node logs its topic but never subscribes. Then work back along the chain from
  [Notebook 2](./2-duckiedrone-tof-architecture.ipynb):

- `ros2 topic info` on the topic the node logged at startup should report one publisher. If the
  hostname in that topic is wrong, nothing publishes on it.

- `ros2 topic hz /<robot>/bottom_tof_driver_node/range` should report about 30 Hz on a physical
  Duckiedrone. If it stays silent, the problem is in the driver or the bridge, not the node. On
  a virtual Duckiedrone, first check that the Duckiematrix is running and the Duckiedrone is
  attached.

- `docker ps` should list `driver-tof-bottom` and `ros2-tof-bottom`. If either is missing,
  nothing publishes the topic.

## Activity 3: make the number move

<a id="activity-3"></a>

Each printed line is the newest reading, unchanged. With the node running, change what the
  downward sensor sees: on a physical Duckiedrone, slide a book slowly under it and take it
  away; on a virtual one, move the Duckiedrone up and down with the Duckiematrix controls.

Three kinds of reading are worth catching:

- **Below the minimum.** A physical Duckiedrone resting on its legs prints something like
  `0.018m`, below the message's `min_range` of `0.04`. The sensor is only specified down to 4
  cm, but it still reports what it measured, and neither the driver nor the bridge clamps the
  value. A virtual Duckiedrone on the ground reads about `0.047m`, so this case appears only on
  a physical one.

- **In range.** Twenty or thirty centimeters, the ordinary case, and the one reading of the
  three that is a usable distance.

- **No reading.** With the motors off, hold the Duckiedrone so the downward sensor faces open
  space with nothing within 3.6 m, for example down a corridor. The line does not go blank: it
  prints `999.000m`, the value the bridge uses when the driver has no distance to report.

Two of the three are not usable distances, yet the node prints all three the same way. That is
  fine for watching the data, but anything that acts on the number has to discard values outside
  `min_range` and `max_range` first, as the [message
  definition](https://github.com/ros2/common_interfaces/blob/jazzy/sensor_msgs/msg/Range.msg)
  asks.

## Going further

Each of these is optional, and each one builds on the node from [Activity 1](#activity-1):

- Discard what the message definition says to discard. Copy `min_range` and `max_range` into
  `ToFReading`, and have `cb_log` print `out of range` instead of the number when `range` falls
  outside them.

- Watch every value rather than one a second: `ros2 topic echo
  /<robot>/bottom_tof_driver_node/range --field range`.

[Notebook 4](./4-all-five-sensors.ipynb) starts the other four sensors and reads all five with
  the same node.